### Flight Cancellation Scraper

- Scrapes the flight cancellation data from the Bureau of Transportation Statistics across 6 major US airports (San Francisco International Airport, LaGuardia Airport, Boston Logan International Airport, Newark Liberty International Airport, Dallas Fort Worth International Airport, Chicago O'Hare International Airport) and 4 airlines (Alaska Airlines, American Airlines, Spirit Airlines, Frontier Airlines) throughout 3 months (December 2024 - February 2025).
- The output CSV (flight_info.csv) will be joined with the weather API data. Every row in the output represents a cancelled flight.

In [1]:
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from io import StringIO

# Establish airports, airlines, and months to scrape
airports = ['SFO', 'LGA', 'BOS', 'EWR', 'DFW', 'ORD']
airlines = ['AS', 'AA', 'NK', 'F9']
months = [("2024", "12"), ("2025", "1"), ("2025", "2")] # December 2024-February 2025

# Map month strings to 0-indexed positions. December = 11, January = 0, February = 1
month_to_index = {"12":11, "1": 0, "2": 1} 

data = [] # Initialize list to collect the DataFrames per iteration

with webdriver.Chrome() as driver:
    url = "https://www.transtats.bts.gov/ontime/Cancellation.aspx"

    for origin in airports:
        print(f"Processing Airport: {origin}") # Track the progress across the airports

        for year, month in months:
            for airline in airlines:
                try:
                    # Reload the page per iteration to reset the checkboxes
                    driver.get(url)
                    time.sleep(2)
    
                    # Select origin airport from the dropdown
                    select_origin = driver.find_element(By.XPATH, f"//select[@id = 'cboAirport']/option[@value = '{origin}']")
                    select_origin.click()
    
                    # Select airline from the dropdown
                    select_airline = driver.find_element(By.XPATH, f"//select[@id = 'cboAirline']/option[@value = '{airline}']")
                    select_airline.click()
    
                    # Select year
                    select_year = driver.find_element(By.XPATH, f"//input[@type = 'checkbox' and @value = '{year}']")
                    if not select_year.is_selected():
                        select_year.click()
    
                    # Select month through the 0-based index
                    month_index = month_to_index[month]
                    select_month = driver.find_element(By.ID, f"chkMonths_{month_index}")
                    if not select_month.is_selected():
                        select_month.click()
    
                    # Select "All Days" to obtain data for every day in that month
                    select_days = driver.find_element(By.ID, "chkAllDays")
                    if not select_days.is_selected():
                        select_days.click()
    
                    # Submit the form
                    submit = driver.find_element(By.ID, "btnSubmit")
                    submit.click()
                    time.sleep(3)

                    # Extract tables
                    html = driver.page_source
                    tables = pd.read_html(StringIO(html))
    
                    # Find the flight data table by finding the "Carrier Code" column
                    for table in tables:
                        if 'Carrier Code' in table.columns:
                            table['Origin'] = origin
                            table['Airline'] = airline
                            table['Month'] = month
                            table['Year'] = year
                            data.append(table)
                            break # Break once the first matching table is found

                # Skip and warn the failed combination (Timeout, element not found, etc)
                except Exception as e:
                    print(f"Skipping {airline} at {origin} {month}/{year}: {e}")

# Save the collected DataFrames into a CSV after merging all results
if data:
    df = pd.concat(data)
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")] # Drop the empty columns from the formatting
    df = df[df["Carrier Code"].str.match('[A-Z][A-Z0-9]$', na = False)] # Remove footer rows. Valid carrier codes begin with a letter, followed by a letter or number
    df.to_csv("flight_info.csv", index = False)

Processing Airport: SFO
Processing Airport: LGA
Processing Airport: BOS
Processing Airport: EWR
Processing Airport: DFW
Processing Airport: ORD
